# 2주차. 데이터 전처리

본 과제에서는 1주차 탐색적 데이터 분석(EDA)을 통해 확인한
데이터의 특성을 바탕으로 데이터 전처리를 수행한다.

전처리 과정에서는 결측치 처리, 이상치 처리, 인코딩,
Scaling을 수행하고 각 과정의 적용 이유와 결과를 분석한다.

In [3]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [10]:
train <- read.csv("train.csv")
test <- read.csv("test.csv")

dim(train)
dim(test)

## 2. 결측치 처리

결측치는 데이터 분석 및 머신러닝 모델 학습 과정에서 문제를 일으킬 수 있으므로,
각 변수에 결측치가 존재하는지 확인한다.

In [14]:
colSums(is.na(train))

ID               Age            Gender           Country 
                0                 0                 0                 0 
             Race Family_Background Radiation_History Iodine_Deficiency 
                0                 0                 0                 0 
            Smoke       Weight_Risk          Diabetes       Nodule_Size 
                0                 0                 0                 0 
       TSH_Result         T4_Result         T3_Result            Cancer 
                0                 0                 0                 0

In [15]:
colSums(is.na(test))

ID               Age            Gender           Country 
                0                 0                 0                 0 
             Race Family_Background Radiation_History Iodine_Deficiency 
                0                 0                 0                 0 
            Smoke       Weight_Risk          Diabetes       Nodule_Size 
                0                 0                 0                 0 
       TSH_Result         T4_Result         T3_Result 
                0                 0                 0

In [16]:
sum(is.na(train))
sum(is.na(test))

[1] 0

[1] 0

In [17]:
num_cols <- c(
  "Age",
  "Nodule_Size",
  "TSH_Result",
  "T4_Result",
  "T3_Result"
)

In [18]:
# 각 수치형 변수의 요약 통계량 확인
summary(train[num_cols])

      Age         Nodule_Size      TSH_Result       T4_Result     
 Min.   :14.00   Min.   :0.000   Min.   : 0.100   Min.   : 4.500  
 1st Qu.:32.00   1st Qu.:1.270   1st Qu.: 2.583   1st Qu.: 6.372  
 Median :51.00   Median :2.521   Median : 5.059   Median : 8.237  
 Mean   :50.86   Mean   :2.508   Mean   : 5.057   Mean   : 8.249  
 3rd Qu.:70.00   3rd Qu.:3.761   3rd Qu.: 7.542   3rd Qu.:10.127  
 Max.   :88.00   Max.   :5.000   Max.   :10.000   Max.   :12.000  
   T3_Result    
 Min.   :0.500  
 1st Qu.:1.255  
 Median :2.004  
 Mean   :2.005  
 3rd Qu.:2.758  
 Max.   :3.500  

## 3. 이상치 처리

In [19]:
# IQR을 이용한 이상치 개수 확인

outlier_count <- sapply(train[num_cols], function(x) {
  Q1 <- quantile(x, 0.25)
  Q3 <- quantile(x, 0.75)
  IQR_value <- Q3 - Q1
  
  lower <- Q1 - 1.5 * IQR_value
  upper <- Q3 + 1.5 * IQR_value
  
  sum(x < lower | x > upper)
})

outlier_count

Age Nodule_Size  TSH_Result   T4_Result   T3_Result 
          0           0           0           0           0

In [20]:
str(train)

'data.frame':	87159 obs. of  16 variables:
 $ ID               : chr  "TRAIN_00000" "TRAIN_00001" "TRAIN_00002" "TRAIN_00003" ...
 $ Age              : int  80 37 71 40 53 86 65 36 67 58 ...
 $ Gender           : chr  "M" "M" "M" "F" ...
 $ Country          : chr  "CHN" "NGA" "CHN" "IND" ...
 $ Race             : chr  "ASN" "ASN" "MDE" "HSP" ...
 $ Family_Background: chr  "Positive" "Positive" "Positive" "Negative" ...
 $ Radiation_History: chr  "Exposed" "Unexposed" "Unexposed" "Unexposed" ...
 $ Iodine_Deficiency: chr  "Sufficient" "Sufficient" "Sufficient" "Sufficient" ...
 $ Smoke            : chr  "Non-Smoker" "Smoker" "Non-Smoker" "Non-Smoker" ...
 $ Weight_Risk      : chr  "Not Obese" "Obese" "Not Obese" "Obese" ...
 $ Diabetes         : chr  "No" "No" "Yes" "No" ...
 $ Nodule_Size      : num  0.65 2.95 2.2 3.37 4.23 ...
 $ TSH_Result       : num  2.785 0.912 0.718 6.846 0.44 ...
 $ T4_Result        : num  6.74 7.3 11.14 10.18 7.19 ...
 $ T3_Result        : num  2.576 2.505 2.38

In [21]:
categorical_cols <- c(
  "Gender",
  "Country",
  "Race",
  "Family_Background",
  "Radiation_History",
  "Iodine_Deficiency",
  "Smoke",
  "Weight_Risk",
  "Diabetes"
)

In [22]:
lapply(train[categorical_cols], unique)

$Gender
[1] "M" "F"

$Country
 [1] "CHN" "NGA" "IND" "USA" "GBR" "BRA" "RUS" "JPN" "KOR" "DEU"

$Race
[1] "ASN" "MDE" "HSP" "CAU" "AFR"

$Family_Background
[1] "Positive" "Negative"

$Radiation_History
[1] "Exposed"   "Unexposed"

$Iodine_Deficiency
[1] "Sufficient" "Deficient" 

$Smoke
[1] "Non-Smoker" "Smoker"    

$Weight_Risk
[1] "Not Obese" "Obese"    

$Diabetes
[1] "No"  "Yes"

In [23]:
sapply(train[categorical_cols], function(x) length(unique(x)))

Gender           Country              Race Family_Background 
                2                10                 5                 2 
Radiation_History Iodine_Deficiency             Smoke       Weight_Risk 
                2                 2                 2                 2 
         Diabetes 
                2

In [24]:
sapply(train[categorical_cols], function(x) length(unique(x)))

Gender           Country              Race Family_Background 
                2                10                 5                 2 
Radiation_History Iodine_Deficiency             Smoke       Weight_Risk 
                2                 2                 2                 2 
         Diabetes 
                2

In [26]:
lapply(categorical_cols, function(col) {
  setdiff(unique(test[[col]]), unique(train[[col]]))
})

[[1]]
character(0)

[[2]]
character(0)

[[3]]
character(0)

[[4]]
character(0)

[[5]]
character(0)

[[6]]
character(0)

[[7]]
character(0)

[[8]]
character(0)

[[9]]
character(0)

In [27]:
# One-Hot Encoding에 사용할 범주형 변수
categorical_cols

[1] "Gender"            "Country"           "Race"             
[4] "Family_Background" "Radiation_History" "Iodine_Deficiency"
[7] "Smoke"             "Weight_Risk"       "Diabetes"

In [28]:
# Train과 Test의 범주형 변수를 factor로 변환
train[categorical_cols] <- lapply(train[categorical_cols], factor)
test[categorical_cols] <- lapply(test[categorical_cols], factor)

# factor 변환 확인
str(train[categorical_cols])

'data.frame':	87159 obs. of  9 variables:
 $ Gender           : Factor w/ 2 levels "F","M": 2 2 2 1 1 1 1 2 2 1 ...
 $ Country          : Factor w/ 10 levels "BRA","CHN","DEU",..: 2 8 2 5 2 10 4 5 8 1 ...
 $ Race             : Factor w/ 5 levels "AFR","ASN","CAU",..: 2 2 5 4 3 1 1 3 4 4 ...
 $ Family_Background: Factor w/ 2 levels "Negative","Positive": 2 2 2 1 1 1 2 2 1 1 ...
 $ Radiation_History: Factor w/ 2 levels "Exposed","Unexposed": 1 2 2 2 2 2 2 2 1 2 ...
 $ Iodine_Deficiency: Factor w/ 2 levels "Deficient","Sufficient": 2 2 2 2 2 2 2 2 1 2 ...
 $ Smoke            : Factor w/ 2 levels "Non-Smoker","Smoker": 1 2 1 1 1 2 1 1 1 2 ...
 $ Weight_Risk      : Factor w/ 2 levels "Not Obese","Obese": 1 2 1 2 1 1 1 1 1 1 ...
 $ Diabetes         : Factor w/ 2 levels "No","Yes": 1 1 2 1 1 1 1 1 1 1 ...


In [29]:
# 범주형 변수에 One-Hot Encoding 적용
encoded_train <- model.matrix(
  ~ . - 1,
  data = train[categorical_cols]
)

# 데이터 구조 확인
dim(encoded_train)
head(encoded_train)

[1] 87159    21

,GenderF,GenderM,CountryCHN,CountryDEU,CountryGBR,CountryIND,CountryJPN,CountryKOR,CountryNGA,CountryRUS,⋯,RaceASN,RaceCAU,RaceHSP,RaceMDE,Family_BackgroundPositive,Radiation_HistoryUnexposed,Iodine_DeficiencySufficient,SmokeSmoker,Weight_RiskObese,DiabetesYes
1,0,1,1,0,0,0,0,0,0,0,⋯,1,0,0,0,1,0,1,0,0,0
2,0,1,0,0,0,0,0,0,1,0,⋯,1,0,0,0,1,1,1,1,1,0
3,0,1,1,0,0,0,0,0,0,0,⋯,0,0,0,1,1,1,1,0,0,1
4,1,0,0,0,0,1,0,0,0,0,⋯,0,0,1,0,0,1,1,0,1,0
5,1,0,1,0,0,0,0,0,0,0,⋯,0,1,0,0,0,1,1,0,0,0
6,1,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,1,1,1,0,0


In [30]:
dim(encoded_train)

[1] 87159    21

In [31]:
encoded_test <- model.matrix(
  ~ . - 1,
  data = test[categorical_cols]
)

dim(encoded_test)

[1] 46204    21

In [32]:
identical(colnames(encoded_train), colnames(encoded_test))

[1] TRUE

In [33]:
# 수치형 변수
num_cols <- c(
  "Age",
  "Nodule_Size",
  "TSH_Result",
  "T4_Result",
  "T3_Result"
)

# Train 데이터의 평균과 표준편차 저장
train_means <- sapply(train[num_cols], mean)
train_sds <- sapply(train[num_cols], sd)

# Train 데이터 표준화
scaled_train_num <- scale(
  train[num_cols],
  center = train_means,
  scale = train_sds
)

# 결과 확인
head(scaled_train_num)

Age,Nodule_Size,TSH_Result,T4_Result,T3_Result
1.34665084,-1.2883744,-0.7941119,-0.6944648,0.6583729
-0.64053072,0.3067641,-1.4487644,-0.4365200,0.5770558
0.93072911,-0.2136550,-1.5165220,1.3336551,0.4337626
-0.50189015,0.5982946,0.6254335,0.8894185,-1.4440147
0.09888567,1.1941995,-1.6137652,-0.4867768,-1.6558535
1.62393198,-0.1228544,1.3691925,0.6058861,-1.6872210


In [34]:
# Test 데이터 표준화
scaled_test_num <- scale(
  test[num_cols],
  center = train_means,
  scale = train_sds
)

# 결과 확인
head(scaled_test_num)

Age,Nodule_Size,TSH_Result,T4_Result,T3_Result
0.09888567,0.3000010,0.4814817,0.4520653,0.8406378
-0.22460900,0.6533133,-0.1144041,-1.0670172,-1.4186529
1.25422379,1.5067477,0.2120081,1.0190184,-0.8593947
0.83830206,1.2292498,0.8447001,0.2477125,0.9470861
1.20801026,0.6047429,0.7140692,0.2254260,1.6707490
0.28373977,1.1807821,1.0206968,-1.6251166,-1.1349051


In [35]:
# Train Scaling 결과의 평균과 표준편차 확인
colMeans(scaled_train_num)
apply(scaled_train_num, 2, sd)

Age   Nodule_Size    TSH_Result     T4_Result     T3_Result 
 1.371974e-16 -2.157776e-16  1.715694e-16 -2.802517e-16  1.087893e-16

Age Nodule_Size  TSH_Result   T4_Result   T3_Result 
          1           1           1           1           1

In [36]:
# 최종 전처리 Train 데이터 생성
train_processed <- data.frame(
  scaled_train_num,
  encoded_train,
  Cancer = train$Cancer
)

# 최종 전처리 Test 데이터 생성
test_processed <- data.frame(
  scaled_test_num,
  encoded_test
)

# 데이터 크기 확인
dim(train_processed)
dim(test_processed)

[1] 87159    27

[1] 46204    26

In [37]:
head(train_processed)
str(train_processed)
colSums(is.na(train_processed))

,Age,Nodule_Size,TSH_Result,T4_Result,T3_Result,GenderF,GenderM,CountryCHN,CountryDEU,CountryGBR,⋯,RaceCAU,RaceHSP,RaceMDE,Family_BackgroundPositive,Radiation_HistoryUnexposed,Iodine_DeficiencySufficient,SmokeSmoker,Weight_RiskObese,DiabetesYes,Cancer
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,1.34665084,-1.2883744,-0.7941119,-0.6944648,0.6583729,0,1,1,0,0,⋯,0,0,0,1,0,1,0,0,0,1
2,-0.64053072,0.3067641,-1.4487644,-0.4365200,0.5770558,0,1,0,0,0,⋯,0,0,0,1,1,1,1,1,0,1
3,0.93072911,-0.2136550,-1.5165220,1.3336551,0.4337626,0,1,1,0,0,⋯,0,0,1,1,1,1,0,0,1,0
4,-0.50189015,0.5982946,0.6254335,0.8894185,-1.4440147,1,0,0,0,0,⋯,0,1,0,0,1,1,0,1,0,0
5,0.09888567,1.1941995,-1.6137652,-0.4867768,-1.6558535,1,0,1,0,0,⋯,1,0,0,0,1,1,0,0,0,1
6,1.62393198,-0.1228544,1.3691925,0.6058861,-1.6872210,1,0,0,0,0,⋯,0,0,0,0,1,1,1,0,0,0


'data.frame':	87159 obs. of  27 variables:
 $ Age                        : num  1.3467 -0.6405 0.9307 -0.5019 0.0989 ...
 $ Nodule_Size                : num  -1.288 0.307 -0.214 0.598 1.194 ...
 $ TSH_Result                 : num  -0.794 -1.449 -1.517 0.625 -1.614 ...
 $ T4_Result                  : num  -0.694 -0.437 1.334 0.889 -0.487 ...
 $ T3_Result                  : num  0.658 0.577 0.434 -1.444 -1.656 ...
 $ GenderF                    : num  0 0 0 1 1 1 1 0 0 1 ...
 $ GenderM                    : num  1 1 1 0 0 0 0 1 1 0 ...
 $ CountryCHN                 : num  1 0 1 0 1 0 0 0 0 0 ...
 $ CountryDEU                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryGBR                 : num  0 0 0 0 0 0 1 0 0 0 ...
 $ CountryIND                 : num  0 0 0 1 0 0 0 1 0 0 ...
 $ CountryJPN                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryKOR                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryNGA                 : num  0 1 0 0 0 0 0 0 1 0 ...
 $ CountryRUS                 : num

Age                 Nodule_Size 
                          0                           0 
                 TSH_Result                   T4_Result 
                          0                           0 
                  T3_Result                     GenderF 
                          0                           0 
                    GenderM                  CountryCHN 
                          0                           0 
                 CountryDEU                  CountryGBR 
                          0                           0 
                 CountryIND                  CountryJPN 
                          0                           0 
                 CountryKOR                  CountryNGA 
                          0                           0 
                 CountryRUS                  CountryUSA 
                          0                           0 
                    RaceASN                     RaceCAU 
                          0                           0 
                    RaceHSP                     RaceMDE 
                          0                           0 
  Family_BackgroundPositive  Radiation_HistoryUnexposed 
                          0                           0 
Iodine_DeficiencySufficient                 SmokeSmoker 
                          0                           0 
           Weight_RiskObese                 DiabetesYes 
                          0                           0 
                     Cancer 
                          0